In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [ ]:
!pip install torch-geometric
!pip install rdkit
!pip install -U bitsandbytes


In [ ]:
!pip -q install -U transformers peft bitsandbytes accelerate sentencepiece pandas


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import torch.nn.functional as F
import os

# Data loading

In [ ]:
from google.colab import drive
import os
import pandas as pd
from pathlib import Path
from google.colab import drive

# 1. Monter Google Drive
drive.mount('/content/drive')

# --- CONFIGURATION DES CHEMINS ---
# Ajustez 'Mon_Projet' selon le nom de votre dossier dans Drive
base_drive_path = Path("/content/drive/MyDrive/Altegrad")



In [ ]:
batch_size = 32
num_epochs = 20
learning_rate = 2e-5
weight_decay = 0.01
warmup_ratio = 0.1
max_smiles_len = 256
max_text_len = 128
temperature = 0.07
num_workers = 4
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
import pickle
import torch

# 1. Load the data

file_path = base_drive_path / 'train_graphs.pkl'

print(f"Loading {file_path}...")
with open(file_path, 'rb') as f:
    graphs = pickle.load(f)

print(f"Successfully loaded {len(graphs)} molecules.\n")

# 2. Inspect the first molecule (index 0)
data = graphs[0]

print("--- Overview of the first Molecule ---")
print(f"Object type: {type(data)}")
print(f"Keys available: {data.keys}\n")

# 3. Inspect specific attributes
print(f"ID: {data.id}")
print(f"Description: {data.description}")

print(f"\nNode Features (data.x):")
print(f"  Shape: {data.x.shape} (Atoms x Features)")
print(f"  First atom features: {data.x[0]}")

print(f"\nConnectivity (data.edge_index):")
print(f"  Shape: {data.edge_index.shape}")
print(f"  Structure (Source -> Target):\n{data.edge_index}")

if hasattr(data, 'edge_attr') and data.edge_attr is not None:
    print(f"\nBond Features (data.edge_attr):")
    print(f"  Shape: {data.edge_attr.shape}")

In [ ]:
import networkx as nx
from torch_geometric.utils import to_networkx
import matplotlib.pyplot as plt

# Convert PyG Data object to a NetworkX graph
# to_undirected=True is usually better for visualization of molecules
G = to_networkx(data, to_undirected=True)

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)  # Compute layout for clear visualization

# Draw the nodes and edges
nx.draw(G, pos,
        with_labels=True,
        node_color='lightblue',
        node_size=500,
        font_size=10,
        edge_color='gray')

plt.title(f"Graph Structure for Molecule ID: {data.id}")
plt.show()

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
import torch

def pyg_to_mol(data):
    """
    Converts a PyTorch Geometric Data object to an RDKit Molecule.
    Assumes data.x[:, 0] contains atomic numbers.
    """
    # 1. Create an empty editable molecule
    mol = Chem.RWMol()

    # 2. Add Atoms
    # We iterate through the node features (data.x)
    atom_indices = []
    for atom_features in data.x:
        # Assumption: The first feature is the Atomic Number (e.g., 6=C, 8=O)
        # We accept float but convert to int
        atomic_num = int(atom_features[0].item())

        # Create RDKit atom and add to molecule
        if atomic_num > 0: # minimal check for valid atoms
            a = Chem.Atom(atomic_num)
            idx = mol.AddAtom(a)
            atom_indices.append(idx)
        else:
            # Fallback for dummy atoms if 0 is used
            a = Chem.Atom(0)
            idx = mol.AddAtom(a)
            atom_indices.append(idx)

    # 3. Add Bonds
    # data.edge_index is shape [2, Num_Edges]
    rows, cols = data.edge_index

    # We loop through edges.
    # Note: PyG graphs are often directed (edges both ways), RDKit needs undirected.
    # We use a set to avoid adding the same bond twice.
    seen_bonds = set()

    for i in range(len(rows)):
        src = int(rows[i].item())
        dst = int(cols[i].item())

        # Sort to handle undirected nature (0-1 is same as 1-0)
        bond_sig = tuple(sorted((src, dst)))

        if bond_sig not in seen_bonds and src != dst:
            mol.AddBond(src, dst, Chem.BondType.SINGLE)
            seen_bonds.add(bond_sig)

    # 4. Sanitize (Detect aromatic rings, valence, etc.)
    try:
        Chem.SanitizeMol(mol)
    except ValueError as e:
        print(f"Warning: Molecule sanitization failed (likely invalid valences): {e}")

    return mol.GetMol()

In [ ]:
# Select a molecule (e.g., index 0)
data = graphs[0]

# Convert to RDKit
mol = pyg_to_mol(data)

# Display
print(f"Molecule ID: {data.id}")
print(f"Description: {data.description}")

# This will render the image in the notebook
display(Draw.MolToImage(mol, size=(400, 300)))

## Preprocessing

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdchem
import torch

def data_to_rdkit_mol(data, x_map, e_map):
    """
    data: torch_geometric.data.Data
    x_map: dict of categorical mappings for node features (from data_utils.py)
    e_map: dict of categorical mappings for edge features (from data_utils.py)
    """

    mol = Chem.RWMol()

    # ---- 1) Add atoms ----
    for i in range(data.x.size(0)):
        feats = data.x[i]

        # Indices in feats must match the order described in your statement:
        # [atomic_num, chirality, degree, formal_charge, num_hs,
        #  num_radical_electrons, hybridization, is_aromatic, is_in_ring]
        atomic_num_idx        = int(feats[0])
        chirality_idx         = int(feats[1])
        degree_idx            = int(feats[2])
        formal_charge_idx     = int(feats[3])
        num_hs_idx            = int(feats[4])
        num_rad_e_idx         = int(feats[5])
        hybridization_idx     = int(feats[6])
        is_aromatic_idx       = int(feats[7])
        is_in_ring_idx        = int(feats[8])

        atomic_num    = x_map["atomic_num"][atomic_num_idx]
        formal_charge = x_map["formal_charge"][formal_charge_idx]
        num_hs        = x_map["num_hs"][num_hs_idx]
        num_rad_e     = x_map["num_radical_electrons"][num_rad_e_idx]

        # RDKit atom
        atom = Chem.Atom(int(atomic_num))
        atom.SetFormalCharge(int(formal_charge))
        atom.SetNumExplicitHs(int(num_hs))
        atom.SetNumRadicalElectrons(int(num_rad_e))

        # Hybridization
        hybridization_str = x_map["hybridization"][hybridization_idx]
        hyb_map = {
            "S":   rdchem.HybridizationType.S,
            "SP":  rdchem.HybridizationType.SP,
            "SP2": rdchem.HybridizationType.SP2,
            "SP3": rdchem.HybridizationType.SP3,
            "SP3D": rdchem.HybridizationType.SP3D,
            "SP3D2": rdchem.HybridizationType.SP3D2,
            "UNSPECIFIED": rdchem.HybridizationType.UNSPECIFIED,
            "OTHER": rdchem.HybridizationType.UNSPECIFIED,
        }
        atom.SetHybridization(hyb_map.get(hybridization_str, rdchem.HybridizationType.UNSPECIFIED))

        # Chirality
        chiral_str = x_map["chirality"][chirality_idx]
        chiral_map = {
            "CHI_UNSPECIFIED": rdchem.ChiralType.CHI_UNSPECIFIED,
            "CHI_TETRAHEDRAL_CW": rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
            "CHI_TETRAHEDRAL_CCW": rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
        }
        atom.SetChiralTag(chiral_map.get(chiral_str, rdchem.ChiralType.CHI_UNSPECIFIED))

        # Aromaticity
        is_aromatic = bool(x_map["is_aromatic"][is_aromatic_idx])
        atom.SetIsAromatic(is_aromatic)

        mol_idx = mol.AddAtom(atom)

    # ---- 2) Add bonds ----
    edge_index = data.edge_index
    edge_attr  = data.edge_attr

    # Here we avoid adding each bond twice (because edges are i->j and j->i)
    added_bonds = set()

    for k in range(edge_index.size(1)):
        u = int(edge_index[0, k])
        v = int(edge_index[1, k])
        if u == v:
            continue

        key = tuple(sorted((u, v)))
        if key in added_bonds:
            continue
        added_bonds.add(key)

        bf = edge_attr[k]
        bond_type_idx   = int(bf[0])
        stereo_idx      = int(bf[1])
        is_conj_idx     = int(bf[2])

        bond_type_str = e_map["bond_type"][bond_type_idx]
        stereo_str    = e_map["stereo"][stereo_idx]
        is_conjugated = bool(e_map["is_conjugated"][is_conj_idx])

        # Bond type mapping (adapt to your e_map)
        bond_type_map = {
            "SINGLE": rdchem.BondType.SINGLE,
            "DOUBLE": rdchem.BondType.DOUBLE,
            "TRIPLE": rdchem.BondType.TRIPLE,
            "AROMATIC": rdchem.BondType.AROMATIC,
        }
        bt = bond_type_map.get(bond_type_str, rdchem.BondType.SINGLE)

        mol.AddBond(u, v, bt)
        bond = mol.GetBondBetweenAtoms(u, v)
        if bond is None:
            continue

        bond.SetIsConjugated(is_conjugated)

        stereo_map = {
            "StereoNone": rdchem.BondStereo.STEREONONE,
            "StereoAny": rdchem.BondStereo.STEREOANY,
            "StereoZ": rdchem.BondStereo.STEREOZ,
            "StereoE": rdchem.BondStereo.STEREOE,
            "StereoCis": rdchem.BondStereo.STEREOZ,
            "StereoTrans": rdchem.BondStereo.STEREOE,
        }
        bond.SetStereo(stereo_map.get(stereo_str, rdchem.BondStereo.STEREONONE))

    # ---- 3) Finalize molecule ----
    mol = mol.GetMol()
    Chem.SanitizeMol(mol)
    return mol


def data_to_smiles(data, x_map, e_map):
    mol = data_to_rdkit_mol(data, x_map, e_map)
    # canonical SMILES
    return Chem.MolToSmiles(mol)


In [ ]:
graphs

In [ ]:
"""
Data loading and processing utilities for molecule-text retrieval.
Includes dataset classes and data loading functions.
"""
from typing import Dict
import pickle

import pandas as pd
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Batch


# =========================================================
# Feature maps for atom and bond attributes
# =========================================================
from typing import Dict, List, Any

x_map: Dict[str, List[Any]] = {
    'atomic_num': list(range(0, 119)),
    'chirality': [
        'CHI_UNSPECIFIED','CHI_TETRAHEDRAL_CW','CHI_TETRAHEDRAL_CCW','CHI_OTHER',
        'CHI_TETRAHEDRAL','CHI_ALLENE','CHI_SQUAREPLANAR','CHI_TRIGONALBIPYRAMIDAL',
        'CHI_OCTAHEDRAL',
    ],
    'degree': list(range(0, 11)),
    'formal_charge': list(range(-5, 7)),
    'num_hs': list(range(0, 9)),
    'num_radical_electrons': list(range(0, 5)),
    'hybridization': [
        'UNSPECIFIED','S','SP','SP2','SP3','SP3D','SP3D2','OTHER',
    ],
    'is_aromatic': [False, True],
    'is_in_ring': [False, True],
}

e_map: Dict[str, List[Any]] = {
    'bond_type': [
        'UNSPECIFIED','SINGLE','DOUBLE','TRIPLE','QUADRUPLE','QUINTUPLE','HEXTUPLE',
        'ONEANDAHALF','TWOANDAHALF','THREEANDAHALF','FOURANDAHALF','FIVEANDAHALF',
        'AROMATIC','IONIC','HYDROGEN','THREECENTER','DATIVEONE','DATIVE','DATIVEL',
        'DATIVER','OTHER','ZERO',
    ],
    'stereo': [
        'STEREONONE','STEREOANY','STEREOZ','STEREOE','STEREOCIS','STEREOTRANS',
    ],
    'is_conjugated': [False, True],
}


# =========================================================
# Load precomputed text embeddings
# =========================================================
def load_id2emb(csv_path: str) -> Dict[str, torch.Tensor]:
    """
    Load precomputed text embeddings from CSV file.

    Args:
        csv_path: Path to CSV file with columns: ID, embedding
                  where embedding is comma-separated floats

    Returns:
        Dictionary mapping ID (str) to embedding tensor
    """
    df = pd.read_csv(csv_path)
    id2emb = {}
    for _, row in df.iterrows():
        id_ = str(row["ID"])
        emb_str = row["embedding"]
        emb_vals = [float(x) for x in str(emb_str).split(',')]
        id2emb[id_] = torch.tensor(emb_vals, dtype=torch.float32)
    return id2emb


# =========================================================
# Load descriptions from preprocessed graphs
# =========================================================
def load_descriptions_from_graphs(graph_path: str) -> Dict[str, str]:
    """
    Load ID to description mapping from preprocessed graph file.

    Args:
        graph_path: Path to .pkl file containing list of pre-saved graphs

    Returns:
        Dictionary mapping ID (str) to description (str)
    """
    with open(graph_path, 'rb') as f:
        graphs = pickle.load(f)

    id2desc = {}
    for graph in graphs:
        id2desc[graph.id] = graph.description

    return id2desc


# =========================================================
# Dataset that loads preprocessed graphs and text embeddings
# =========================================================
class PreprocessedGraphDataset(Dataset):
    """
    Dataset that loads pre-saved molecule graphs with optional text embeddings.

    Args:
        graph_path: Path to .pkl file containing list of pre-saved graphs
        emb_dict: Dictionary mapping ID to text embedding tensors (optional)
    """
    def __init__(self, graph_path: str, emb_dict: Dict[str, torch.Tensor] = None):
        print(f"Loading graphs from: {graph_path}")
        with open(graph_path, 'rb') as f:
            self.graphs = pickle.load(f)
        self.emb_dict = emb_dict
        self.ids = [g.id for g in self.graphs]
        print(f"Loaded {len(self.graphs)} graphs")

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, idx):
        graph = self.graphs[idx]
        if self.emb_dict is not None:
            id_ = graph.id
            text_emb = self.emb_dict[id_]
            return graph, text_emb
        else:
            return graph






In [ ]:
train_dataset = PreprocessedGraphDataset("/content/drive/MyDrive/Altegrad/train_graphs.pkl")

all_smiles = []
all_ids = []

for data in train_dataset:
    smi = data_to_smiles(data, x_map, e_map)
    all_smiles.append(smi)
    all_ids.append(data.id)


In [ ]:
all_smiles

Sanity check

In [ ]:
#from rdkit import Chem
#
#for s in all_smiles:
#    mol = Chem.MolFromSmiles(s)
#    print("INVALID" if mol is  None else "",
#          Chem.MolToSmiles(mol) if mol is None else "")


# Dataset

In [ ]:
# create dataset with 2 columns: smiles and description
from datasets import Dataset

dataset = pd.DataFrame({
    "smiles": all_smiles,
    "description": [data.description for data in train_dataset]})


ds_train = Dataset.from_pandas(dataset)
ds_train

In [ ]:
ds_train[0]

In [ ]:
val_dataset = PreprocessedGraphDataset("/content/drive/MyDrive/Altegrad/validation_graphs.pkl")

all_smiles_val = []
all_ids_val = []

for data in val_dataset:
    smi = data_to_smiles(data, x_map, e_map)
    all_smiles_val.append(smi)
    all_ids_val.append(data.id)

dataset_val = pd.DataFrame({
    "smiles": all_smiles_val,
    "description": [data.description for data in val_dataset]})
ds_val = Dataset.from_pandas(dataset_val)

In [ ]:
import pandas as pd
from datasets import Dataset
from torch.utils.data import DataLoader

# 1. Load Data
test_dataset = PreprocessedGraphDataset("/content/drive/MyDrive/Altegrad/test_graphs.pkl")

all_smiles_test = []
all_ids_test = []

print("Converting graphs to SMILES...")
for data in test_dataset:
    smi = data_to_smiles(data, x_map, e_map)

    # [FIX] Changed 'all_smiles_val' to 'all_smiles_test'
    all_smiles_test.append(smi)
    all_ids_test.append(data.id)

# 2. Create DataFrame
# We add 'description' as empty strings so the collate_fn doesn't break
dataset_test = pd.DataFrame({
    "id": all_ids_test,
    "smiles": all_smiles_test,
    "description": [""] * len(all_smiles_test)
})

# 3. Create HF Dataset and Loader
ds_test = Dataset.from_pandas(dataset_test)

In [ ]:
class MolTextDataset(Dataset):
    def __init__(self, hf_dataset, mol_tokenizer, text_tokenizer, max_len, mode='train'):
        """
        Args:
            hf_dataset: Hugging Face Dataset object (ds_train or ds_val)
            mode: 'train' enables SMILES augmentation, 'val' disables it.
        """
        self.data = hf_dataset
        self.mol_tokenizer = mol_tokenizer
        self.text_tokenizer = text_tokenizer
        self.max_len = max_len
        self.mode = mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Hugging Face Dataset returns a dict: {'smiles': ..., 'description': ...}
        row = self.data[idx]
        smiles = row['smiles']
        text = row['description']

        # Apply Augmentation ONLY for training
        if self.mode == 'train':
            smiles = randomize_smiles(smiles)

        # Tokenize Text
        text_inputs = self.text_tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors="pt"
        )

        # Tokenize SMILES
        mol_inputs = self.mol_tokenizer(
            smiles,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'mol_input_ids': mol_inputs['input_ids'].squeeze(0),
            'mol_attention_mask': mol_inputs['attention_mask'].squeeze(0),
            'text_input_ids': text_inputs['input_ids'].squeeze(0),
            'text_attention_mask': text_inputs['attention_mask'].squeeze(0)
        }

# Training

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from transformers import get_cosine_schedule_with_warmup
from torch.optim import AdamW
from rdkit import Chem
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import matplotlib.pyplot as plt # Added for plotting
from rdkit import RDLogger

# Disable RDKit logs (warnings and errors)
lg = RDLogger.logger()
lg.setLevel(RDLogger.CRITICAL)


# Mount Google Drive
drive.mount('/content/drive')

# Create a folder for your checkpoints
CHECKPOINT_DIR = '/content/drive/MyDrive/Altegrad/MolText_Retrieval_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ==========================================
# 1. Configuration & Hyperparameters
# ==========================================
CONFIG = {
    "mol_model_name": "ibm/MoLFormer-XL-both-10pct",
    "text_model_name": "NeuML/pubmedbert-base-embeddings",
    "max_length": 128,
    "projection_dim": 256,
    # MAXIMIZE THIS. If OOM, lower it. High physical batch > Gradient Accumulation
    "batch_size": 128,
    "epochs": 15,
    "lr_encoder": 2e-5,    # Slower for pre-trained weights
    "lr_head": 1e-4,       # Faster for random projection heads
    "temperature": 0.07,
    "gradient_accumulation_steps": 4,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42
}


# Set seeds for reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

def load_latest_checkpoint(path_dir, model, optimizer, scheduler, history=None):
    last_ckpt_path = os.path.join(path_dir, "last_checkpoint.pt")

    if not os.path.exists(last_ckpt_path):
        print("No 'last_checkpoint.pt' found. Starting from scratch.")
        return 0, 0.0

    print(f"Resuming from: {last_ckpt_path}")
    checkpoint = torch.load(last_ckpt_path, map_location=CONFIG['device'])

    # 1. ALWAYS Load Model Weights (This works because architecture is same)
    model.load_state_dict(checkpoint['model_state_dict'])

    # 2. TRY to Load Optimizer
    try:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print("✅ Optimizer state loaded successfully.")
    except ValueError as e:
        # This catches the "param_groups" mismatch
        print(f"⚠️ WARNING: Optimizer structure changed (New LRs?).")
        print("   -> Loading MODEL weights only.")
        print("   -> Resetting Optimizer & Scheduler state.")
        # We do NOT load optimizer/scheduler state here, effectively resetting them

        # Optional: You might want to lower the epoch count if you treat this as a "new phase"
        # But generally, keeping the epoch is fine.

    if history is not None and 'history' in checkpoint:
        history.update(checkpoint['history'])

    start_epoch = checkpoint['epoch'] + 1
    best_r1 = checkpoint.get('best_r1', 0.0)

    print(f"Resuming at Epoch {start_epoch} with Best R@1: {best_r1:.4f}")
    return start_epoch, best_r1

# ==========================================
# 2. Data Augmentation & Dataset
# ==========================================

def randomize_smiles(smiles):
    """
    Winning Strategy: Randomize SMILES to force structure learning
    instead of string memorization.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return smiles
        # Generate random SMILES
        return Chem.MolToSmiles(mol, doRandom=True, canonical=False)
    except:
        return smiles

class MolTextDataset(Dataset):
    def __init__(self, data, mol_tokenizer, text_tokenizer, max_len, mode='train'):
        self.data = data
        self.mol_tokenizer = mol_tokenizer
        self.text_tokenizer = text_tokenizer
        self.max_len = max_len
        self.mode = mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        smiles = row['smiles']
        text = row['description']

        # Mixed Strategy: 10% Random, 90% Canonical during training
        if self.mode == 'train':
            if np.random.rand() > 0.9:
                smiles = self.randomize_smiles(smiles)
            # else: keep canonical

        text_inputs = self.text_tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt")
        mol_inputs = self.mol_tokenizer(smiles, truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt")

        return {
            'mol_input_ids': mol_inputs['input_ids'].squeeze(0),
            'mol_attention_mask': mol_inputs['attention_mask'].squeeze(0),
            'text_input_ids': text_inputs['input_ids'].squeeze(0),
            'text_attention_mask': text_inputs['attention_mask'].squeeze(0)
        }

    def randomize_smiles(self, smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None: return smiles
            return Chem.MolToSmiles(mol, doRandom=True, canonical=False)
        except:
            return smiles

# ==========================================
# 3. Model Architecture
# ==========================================

class AttentivePooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, last_hidden_state, attention_mask):
        # weights: [Batch, SeqLen, 1]
        weights = self.attention(last_hidden_state)

        # Mask padding tokens so they don't contribute
        weights = weights.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)

        # Softmax to get probabilities
        scores = F.softmax(weights, dim=1)

        # Weighted sum
        return torch.sum(last_hidden_state * scores, dim=1)

class ContrastiveModel(nn.Module):
    def __init__(self):
        super().__init__()

        # --- Encoders ---
        # Trust remote code is needed for MolFormer
        self.mol_encoder = AutoModel.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
        self.text_encoder = AutoModel.from_pretrained(CONFIG['text_model_name'])


        mol_hidden_size = self.mol_encoder.config.hidden_size
        text_hidden_size = self.text_encoder.config.hidden_size

        self.mol_pool = AttentivePooling(mol_hidden_size)
        self.text_pool = AttentivePooling(text_hidden_size)

        # --- Projection Heads ---
        # Winning Strategy: Project to lower dim with non-linearity
        self.mol_proj = nn.Sequential(
            nn.Linear(mol_hidden_size, mol_hidden_size),
            nn.BatchNorm1d(mol_hidden_size),
            nn.GELU(),
            nn.Linear(mol_hidden_size, CONFIG['projection_dim'])
        )

        self.text_proj = nn.Sequential(
            nn.Linear(text_hidden_size, text_hidden_size),
            nn.BatchNorm1d(text_hidden_size),
            nn.GELU(),
            nn.Linear(text_hidden_size, CONFIG['projection_dim'])
        )

        # --- Learnable Temperature ---
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / CONFIG['temperature']))


    def forward(self, batch):
        # 1. Encode Molecule
        mol_out = self.mol_encoder(input_ids=batch['mol_input_ids'], attention_mask=batch['mol_attention_mask'])
        # Use Attentive Pooling instead of CLS
        mol_embeddings = self.mol_pool(mol_out.last_hidden_state, batch['mol_attention_mask'])

        # 2. Encode Text
        text_out = self.text_encoder(input_ids=batch['text_input_ids'], attention_mask=batch['text_attention_mask'])
        # Use Attentive Pooling instead of Mean
        text_embeddings = self.text_pool(text_out.last_hidden_state, batch['text_attention_mask'])

        # 3. Project and Normalize
        mol_feat = self.mol_proj(mol_embeddings)
        text_feat = self.text_proj(text_embeddings)

        mol_feat = F.normalize(mol_feat, p=2, dim=1)
        text_feat = F.normalize(text_feat, p=2, dim=1)

        return mol_feat, text_feat

# ==========================================
# 4. Training Loop
# ==========================================

import glob
import re

def save_checkpoint(model, optimizer, scheduler, epoch, loss, path):
    """Saves the model and training state to Google Drive."""
    print(f"Saving checkpoint for epoch {epoch}...")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss,
    }, path)
    print(f"Checkpoint saved: {path}")




# ==========================================
# 2. Helper Functions (Metrics & Plotting)
# ==========================================

def compute_metrics(model, dataloader, device):
    model.eval()
    mol_feats = []
    text_feats = []

    print("Computing validation embeddings...")
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Embeddings"):
            batch = {k: v.to(device) for k, v in batch.items()}
            # Note: We don't need autocast here usually, or can use it for speed
            mol_f, text_f = model(batch)

            # Move to CPU immediately to save GPU memory
            mol_feats.append(mol_f.cpu())
            text_feats.append(text_f.cpu())

    # Concatenate on CPU
    mol_feats = torch.cat(mol_feats, dim=0)
    text_feats = torch.cat(text_feats, dim=0)

    # 5. SAFETY: Matrix Mult on CPU
    # This prevents OOM if N > 10,000
    print("Computing similarity matrix...")
    sim_matrix = torch.matmul(mol_feats, text_feats.T)

    n_samples = sim_matrix.shape[0]

    # argsort on CPU is fast enough for <50k samples
    sorted_indices = torch.argsort(sim_matrix, dim=1, descending=True)
    targets = torch.arange(n_samples).unsqueeze(1) # CPU tensor

    ranks = (sorted_indices == targets).nonzero(as_tuple=True)[1]
    ranks = ranks.float() + 1

    r1 = (ranks == 1).float().mean().item()
    mrr = (1 / ranks).mean().item()

    return r1, mrr

def plot_training_history(history, save_dir):
    """Plots Loss, R@1, and MRR and saves image."""
    epochs = range(1, len(history['train_loss']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot Loss
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    # Plot Metrics
    ax2.plot(epochs, history['r1'], 'g-o', label='R@1')
    ax2.plot(epochs, history['mrr'], 'm-s', label='MRR')
    ax2.set_title('Retrieval Metrics (SMILES -> Text)')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Score')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    save_path = os.path.join(save_dir, 'training_history.png')
    plt.savefig(save_path)
    print(f"Plot saved to {save_path}")
    plt.close()

# ==========================================
# 3. The Main Loop (Modified)
# ==========================================

def train():
    # --- Setup ---
    print("Loading Tokenizers...")

    # 1. CRITICAL FIX: Prevent tokenizer deadlocks when using multiple workers
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    mol_tokenizer = AutoTokenizer.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
    text_tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model_name'])

    # --- Datasets ---
    train_dataset = MolTextDataset(ds_train, mol_tokenizer, text_tokenizer, CONFIG['max_length'], mode='train')
    val_dataset = MolTextDataset(ds_val, mol_tokenizer, text_tokenizer, CONFIG['max_length'], mode='val')

    # --- DataLoaders (Optimized) ---
    train_loader = DataLoader(
        train_dataset + val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        drop_last=True,
        num_workers=4,           # <--- Parallel loading (CPU cores)
        pin_memory=True,         # <--- Faster CPU -> GPU transfer
        persistent_workers=True  # <--- Keeps workers alive between epochs
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=4,           # <--- Parallel loading
        pin_memory=True          # <--- Faster transfer
    )

    model = ContrastiveModel().to(CONFIG['device'])
    optimizer = AdamW([
        # Group 1: The Pre-trained Encoders (Sensitive, keep LR low)
        {
            'params': model.mol_encoder.parameters(),
            'lr': CONFIG['lr_encoder']
        },
        {
            'params': model.text_encoder.parameters(),
            'lr': CONFIG['lr_encoder']
        },

        # Group 2: The Random Projection Heads (Robust, need high LR)
        {
            'params': model.mol_proj.parameters(),
            'lr': CONFIG['lr_head']
        },
        {
            'params': model.text_proj.parameters(),
            'lr': CONFIG['lr_head']
        },

        # Group 3: The Learnable Temperature (Needs to move fast to find equilibrium)
        {
            'params': [model.logit_scale],
            'lr': CONFIG['lr_head']
        }
    ], weight_decay=0.01)

    num_update_steps_per_epoch = len(train_loader) // CONFIG['gradient_accumulation_steps']
    total_steps = num_update_steps_per_epoch * CONFIG['epochs']

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    history = {'train_loss': [], 'val_loss': [], 'r1': [], 'mrr': []}

    # --- RELOADING LOGIC ---
    start_epoch, best_r1 = load_latest_checkpoint(CHECKPOINT_DIR, model, optimizer, scheduler, history)

    print(f"Physical Batch: {CONFIG['batch_size']} | Effective Batch: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")

    optimizer.zero_grad()

    # ### FIX 1: Start from 'start_epoch', not 0
    for epoch in range(start_epoch, CONFIG['epochs']):

        # --- Train ---
        model.train()
        train_loss_accum = 0

        loop = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1} [Train]")

        for step, batch in loop:
            batch = {k: v.to(CONFIG['device']) for k, v in batch.items()}

            mol_feat, text_feat = model(batch)

            with torch.no_grad():
                model.logit_scale.clamp_(max=4.605)
            logit_scale = model.logit_scale.exp()

            logits = logit_scale * (mol_feat @ text_feat.t())
            labels = torch.arange(len(mol_feat), device=CONFIG['device'])
            loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2

            loss = loss / CONFIG['gradient_accumulation_steps']
            loss.backward()

            if (step + 1) % CONFIG['gradient_accumulation_steps'] == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            current_loss = loss.item() * CONFIG['gradient_accumulation_steps']
            train_loss_accum += current_loss
            loop.set_postfix(loss=current_loss)

        avg_train_loss = train_loss_accum / len(train_loader)

        # --- Validation ---
        model.eval()
        val_loss_accum = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(CONFIG['device']) for k, v in batch.items()}
                mol_feat, text_feat = model(batch)
                logit_scale = model.logit_scale.exp()
                logits = logit_scale * (mol_feat @ text_feat.t())
                labels = torch.arange(len(mol_feat), device=CONFIG['device'])
                loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2
                val_loss_accum += loss.item()

        avg_val_loss = val_loss_accum / len(val_loader)

        r1, mrr = compute_metrics(model, val_loader, CONFIG['device'])

        print(f"\nEpoch {epoch+1} Summary:")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"R@1: {r1:.4f} | MRR: {mrr:.4f}")

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['r1'].append(r1)
        history['mrr'].append(mrr)

        plot_training_history(history, CHECKPOINT_DIR)

        # ### FIX 2 & 3: Correct Saving Logic

        # Save Best Model (only if improved)
        if r1 > best_r1:
            best_r1 = r1
            print(f">>> New Best Model (R@1: {best_r1:.4f})! Saving 'best_model.pt'...")
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_model.pt"))

        # Save Last Checkpoint (Always overwrite this specific file)
        # We also MUST save 'best_r1' so we don't lose track of it on resume
        print("Saving 'last_checkpoint.pt'...")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history,
            'best_r1': best_r1  # <--- Crucial!
        }, os.path.join(CHECKPOINT_DIR, "last_checkpoint.pt"))

if __name__ == "__main__":
    train()

# inference (without reranking)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# ==========================================
# 1. Setup & Config
# ==========================================
CHECKPOINT_DIR = '/content/drive/MyDrive/Altegrad/MolText_Retrieval_Checkpoints'

# Must match training config exactly
CONFIG = {
    "mol_model_name": "ibm/MoLFormer-XL-both-10pct",
    "text_model_name": "NeuML/pubmedbert-base-embeddings",
    "max_length": 128,
    "projection_dim": 256,
    "batch_size": 192,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "temperature": 0.07,
    "seed": 42
}

# ==========================================
# 2. Model Components (Must Match Training!)
# ==========================================

class AttentivePooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, last_hidden_state, attention_mask):
        # weights: [Batch, SeqLen, 1]
        weights = self.attention(last_hidden_state)

        # Mask padding tokens so they don't contribute
        weights = weights.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)

        # Softmax to get probabilities
        scores = F.softmax(weights, dim=1)

        # Weighted sum
        return torch.sum(last_hidden_state * scores, dim=1)

class ContrastiveModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoders
        self.mol_encoder = AutoModel.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
        self.text_encoder = AutoModel.from_pretrained(CONFIG['text_model_name'])

        mol_dim = self.mol_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size

        # Pooling Layers (Added this to match training)
        self.mol_pool = AttentivePooling(mol_dim)
        self.text_pool = AttentivePooling(text_dim)

        # Projections
        self.mol_proj = nn.Sequential(
            nn.Linear(mol_dim, mol_dim), nn.BatchNorm1d(mol_dim), nn.GELU(),
            nn.Linear(mol_dim, CONFIG['projection_dim'])
        )
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, text_dim), nn.BatchNorm1d(text_dim), nn.GELU(),
            nn.Linear(text_dim, CONFIG['projection_dim'])
        )

        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / CONFIG['temperature']))

    def forward(self, batch):
        # This is strictly for training, but defined here to complete the class structure
        mol_out = self.mol_encoder(input_ids=batch['mol_input_ids'], attention_mask=batch['mol_attention_mask'])
        mol_embeddings = self.mol_pool(mol_out.last_hidden_state, batch['mol_attention_mask'])

        text_out = self.text_encoder(input_ids=batch['text_input_ids'], attention_mask=batch['text_attention_mask'])
        text_embeddings = self.text_pool(text_out.last_hidden_state, batch['text_attention_mask'])

        mol_feat = self.mol_proj(mol_embeddings)
        text_feat = self.text_proj(text_embeddings)

        return mol_feat, text_feat

# ==========================================
# 3. Specialized Datasets
# ==========================================

class TextBankDataset(Dataset):
    """Stores training descriptions to build the index."""
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'raw_text': text
        }

class TestMolDataset(Dataset):
    """Stores test molecules (SMILES only) + IDs."""
    def __init__(self, ids, smiles_list, tokenizer, max_len):
        self.ids = ids
        self.smiles_list = smiles_list
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        smiles = str(self.smiles_list[idx])
        inputs = self.tokenizer(
            smiles, truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'id': self.ids[idx],
            'raw_smiles': smiles
        }

# ==========================================
# 4. Main Retrieval Pipeline
# ==========================================

def generate_retrieval_submission(ds_train, ds_test, output_file):
    print("--- 1. Loading Model & Weights ---")
    mol_tokenizer = AutoTokenizer.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
    text_tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model_name'])

    model = ContrastiveModel().to(CONFIG['device'])

    # Load Weights
    ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
    print(f"Loading from {ckpt_path}...")

    state_dict = torch.load(ckpt_path, map_location=CONFIG['device'])

    # Handle potential key mismatch (if model was saved as a nested dict)
    if 'model_state_dict' in state_dict:
        state_dict = state_dict['model_state_dict']

    model.load_state_dict(state_dict)
    model.eval()

    # =========================================================
    # A. Build the Text Bank (From Training Data)
    # =========================================================
    print("\n--- 2. Building Text Bank (Indexing Training Descriptions) ---")

    bank_dataset = TextBankDataset(ds_train['description'], text_tokenizer, CONFIG['max_length'])
    bank_loader = DataLoader(bank_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    bank_embs = []
    bank_texts = []

    with torch.no_grad():
        for batch in tqdm(bank_loader, desc="Indexing Bank"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])

            bank_texts.extend(batch['raw_text'])

            # Encode Text
            outputs = model.text_encoder(input_ids=input_ids, attention_mask=attention_mask)

            # Use Attentive Pooling (Matches Training)
            txt_vec = model.text_pool(outputs.last_hidden_state, attention_mask)
            txt_z = model.text_proj(txt_vec)

            bank_embs.append(F.normalize(txt_z, dim=-1).cpu())

    bank_embs = torch.cat(bank_embs, dim=0)
    print(f"Bank Built: {bank_embs.shape} embeddings.")

    # =========================================================
    # B. Encode Test Molecules (From Test Data)
    # =========================================================
    print("\n--- 3. Encoding Test Molecules ---")

    test_dataset = TestMolDataset(ds_test['id'], ds_test['smiles'], mol_tokenizer, CONFIG['max_length'])
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    test_mol_embs = []
    test_ids = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Encoding Test"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])

            test_ids.extend(batch['id'])

            # Encode Molecule
            outputs = model.mol_encoder(input_ids=input_ids, attention_mask=attention_mask)

            # Use Attentive Pooling (Matches Training)
            mol_vec = model.mol_pool(outputs.last_hidden_state, attention_mask)
            mol_z = model.mol_proj(mol_vec)

            test_mol_embs.append(F.normalize(mol_z, dim=-1).cpu())

    test_mol_embs = torch.cat(test_mol_embs, dim=0)
    print(f"Test Mols Encoded: {test_mol_embs.shape}")

    # =========================================================
    # C. Similarity Search & Save
    # =========================================================
    print("\n--- 4. Running Similarity Search & Saving ---")

    # Matrix Multiply: (Num_Test x Dim) @ (Dim x Num_Train) -> (Num_Test x Num_Train)
    scores = torch.matmul(test_mol_embs, bank_embs.T)

    # For each test molecule, find index of max score in the bank
    best_indices = torch.argmax(scores, dim=1)

    # Compile Results
    results = []
    for i, best_idx in enumerate(best_indices):
        results.append({
            "ID": test_ids[i].item() if isinstance(test_ids[i], torch.Tensor) else test_ids[i],
            "description": bank_texts[best_idx.item()]
        })

    df = pd.DataFrame(results)
    df.to_csv(output_file, index=False)
    print(f"✅ Submission saved to {output_file}")
    print(df.head())

# ==========================================
# Run It
# ==========================================
if __name__ == "__main__":

    # Ensure ds_train and ds_test are defined in your environment before running this!
    # Example:
    # ds_train = pd.read_csv("path/to/train.csv")
    # ds_test = pd.read_csv("path/to/test.csv")

    OUTPUT_FILE = '/content/drive/MyDrive/Altegrad/submission.csv'
    generate_retrieval_submission(ds_train, ds_test, OUTPUT_FILE)

# inference with confidence

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# ==========================================
# 1. Setup & Config
# ==========================================
CHECKPOINT_DIR = '/content/drive/MyDrive/Altegrad/MolText_Retrieval_Checkpoints'

# Must match training config exactly
CONFIG = {
    "mol_model_name": "ibm/MoLFormer-XL-both-10pct",
    "text_model_name": "NeuML/pubmedbert-base-embeddings",
    "max_length": 128,
    "projection_dim": 256,
    "batch_size": 192,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "temperature": 0.07,
    "seed": 42
}

# ==========================================
# 2. Model Components (Must Match Training!)
# ==========================================

class AttentivePooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, last_hidden_state, attention_mask):
        # weights: [Batch, SeqLen, 1]
        weights = self.attention(last_hidden_state)

        # Mask padding tokens so they don't contribute
        weights = weights.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)

        # Softmax to get probabilities
        scores = F.softmax(weights, dim=1)

        # Weighted sum
        return torch.sum(last_hidden_state * scores, dim=1)

class ContrastiveModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoders
        self.mol_encoder = AutoModel.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
        self.text_encoder = AutoModel.from_pretrained(CONFIG['text_model_name'])

        mol_dim = self.mol_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size

        # Pooling Layers (Added this to match training)
        self.mol_pool = AttentivePooling(mol_dim)
        self.text_pool = AttentivePooling(text_dim)

        # Projections
        self.mol_proj = nn.Sequential(
            nn.Linear(mol_dim, mol_dim), nn.BatchNorm1d(mol_dim), nn.GELU(),
            nn.Linear(mol_dim, CONFIG['projection_dim'])
        )
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, text_dim), nn.BatchNorm1d(text_dim), nn.GELU(),
            nn.Linear(text_dim, CONFIG['projection_dim'])
        )

        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / CONFIG['temperature']))

    def forward(self, batch):
        # This is strictly for training, but defined here to complete the class structure
        mol_out = self.mol_encoder(input_ids=batch['mol_input_ids'], attention_mask=batch['mol_attention_mask'])
        mol_embeddings = self.mol_pool(mol_out.last_hidden_state, batch['mol_attention_mask'])

        text_out = self.text_encoder(input_ids=batch['text_input_ids'], attention_mask=batch['text_attention_mask'])
        text_embeddings = self.text_pool(text_out.last_hidden_state, batch['text_attention_mask'])

        mol_feat = self.mol_proj(mol_embeddings)
        text_feat = self.text_proj(text_embeddings)

        return mol_feat, text_feat

# ==========================================
# 3. Specialized Datasets
# ==========================================

class TextBankDataset(Dataset):
    """Stores training descriptions to build the index."""
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'raw_text': text
        }

class TestMolDataset(Dataset):
    """Stores test molecules (SMILES only) + IDs."""
    def __init__(self, ids, smiles_list, tokenizer, max_len):
        self.ids = ids
        self.smiles_list = smiles_list
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        smiles = str(self.smiles_list[idx])
        inputs = self.tokenizer(
            smiles, truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'id': self.ids[idx],
            'raw_smiles': smiles
        }

# ==========================================
# 4. Main Retrieval Pipeline
# ==========================================

def generate_retrieval_submission(ds_train, ds_test, output_file):
    print("--- 1. Loading Model & Weights ---")
    mol_tokenizer = AutoTokenizer.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
    text_tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model_name'])

    model = ContrastiveModel().to(CONFIG['device'])

    # Load Weights
    ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
    print(f"Loading from {ckpt_path}...")

    state_dict = torch.load(ckpt_path, map_location=CONFIG['device'])

    # Handle potential key mismatch (if model was saved as a nested dict)
    if 'model_state_dict' in state_dict:
        state_dict = state_dict['model_state_dict']

    model.load_state_dict(state_dict)
    model.eval()

    # =========================================================
    # A. Build the Text Bank (From Training Data)
    # =========================================================
    print("\n--- 2. Building Text Bank (Indexing Training Descriptions) ---")

    bank_dataset = TextBankDataset(ds_train['description'], text_tokenizer, CONFIG['max_length'])
    bank_loader = DataLoader(bank_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    bank_embs = []
    bank_texts = []

    with torch.no_grad():
        for batch in tqdm(bank_loader, desc="Indexing Bank"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])

            bank_texts.extend(batch['raw_text'])

            # Encode Text
            outputs = model.text_encoder(input_ids=input_ids, attention_mask=attention_mask)

            # Use Attentive Pooling (Matches Training)
            txt_vec = model.text_pool(outputs.last_hidden_state, attention_mask)
            txt_z = model.text_proj(txt_vec)

            bank_embs.append(F.normalize(txt_z, dim=-1).cpu())

    bank_embs = torch.cat(bank_embs, dim=0)
    print(f"Bank Built: {bank_embs.shape} embeddings.")

    # =========================================================
    # B. Encode Test Molecules (From Test Data)
    # =========================================================
    print("\n--- 3. Encoding Test Molecules ---")

    test_dataset = TestMolDataset(ds_test['id'], ds_test['smiles'], mol_tokenizer, CONFIG['max_length'])
    test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    test_mol_embs = []
    test_ids = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Encoding Test"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            attention_mask = batch['attention_mask'].to(CONFIG['device'])

            test_ids.extend(batch['id'])

            # Encode Molecule
            outputs = model.mol_encoder(input_ids=input_ids, attention_mask=attention_mask)

            # Use Attentive Pooling (Matches Training)
            mol_vec = model.mol_pool(outputs.last_hidden_state, attention_mask)
            mol_z = model.mol_proj(mol_vec)

            test_mol_embs.append(F.normalize(mol_z, dim=-1).cpu())

    test_mol_embs = torch.cat(test_mol_embs, dim=0)
    print(f"Test Mols Encoded: {test_mol_embs.shape}")



# =========================================================
    # C. Similarity Search & Save (MODIFIED)
    # =========================================================
    print("\n--- 4. Running Similarity Search & Saving ---")

    # Matrix Multiply: (Num_Test x Dim) @ (Dim x Num_Train) -> (Num_Test x Num_Train)
    # Since vectors are normalized, this result IS the Cosine Similarity (-1.0 to 1.0)
    scores = torch.matmul(test_mol_embs, bank_embs.T)

    # torch.max returns a named tuple (values, indices)
    best_scores, best_indices = torch.max(scores, dim=1)

    # Compile Results
    results = []
    for i in range(len(best_indices)):
        idx = best_indices[i].item()
        score = best_scores[i].item() # This is your similarity metric

        results.append({
            "ID": test_ids[i].item() if isinstance(test_ids[i], torch.Tensor) else test_ids[i],
            "description": bank_texts[idx],
            "confidence_score": round(score, 4) # Add the metric here
        })

    df = pd.DataFrame(results)


    df.to_csv(output_file, index=False)
    print(f"✅ Submission saved to {output_file}")

    # =========================================================
    # D. Analyze the Metric
    # =========================================================
    print(f"\n--- Metric Statistics ---")
    print(f"Mean Similarity: {df['confidence_score'].mean():.4f}")
    print(f"Min Similarity:  {df['confidence_score'].min():.4f}")
    print(f"Max Similarity:  {df['confidence_score'].max():.4f}")

    # Visualization
    plt.figure(figsize=(10, 6))
    plt.hist(df['confidence_score'], bins=50, color='skyblue', edgecolor='black')
    plt.title('Distribution of Retrieval Confidence (Cosine Similarity)')
    plt.xlabel('Cosine Similarity Score')
    plt.ylabel('Count')
    plt.grid(True, alpha=0.3)
    plt.savefig('confidence_distribution.png')
    print("Distribution plot saved to 'confidence_distribution.png'")

    print(df.head())


if __name__ == "__main__":


    OUTPUT_FILE = '/content/drive/MyDrive/Altegrad/submission.csv'
    generate_retrieval_submission(ds_train, ds_test, OUTPUT_FILE)

In [ ]:
df_rag = pd.read_csv('/content/sub_rag.csv')
df_rag.rename(columns={"description":"desc_rag"}, inplace=True)
df_rag.head()

In [ ]:
df_rag.describe()

In [ ]:
df_gen = pd.read_csv("/content/sub_gen.csv")
df_gen.rename(columns={"description":"desc_gen"}, inplace=True)
df_gen.head()

In [ ]:
# merge the two dataframes on ID,
df_merged = pd.merge(df_rag, df_gen, on='ID')
df_merged.head()

In [ ]:
import numpy as np

# Logique : Si score > 0.83, on prend 'desc_rag', sinon on prend 'description' (ou autre colonne)
df_merged["description"] = np.where(
    df_merged["confidence_score"] > 0.83,  # Condition
    df_merged["desc_rag"],                 # Valeur si VRAI
    df_merged["desc_gen"]               # Valeur si FAUX
)

df_merged = df_merged[["ID", "description"]]
df_merged

In [ ]:
df_merged.to_csv("mix.csv")

# Reranker

## step 1: Hard negative mining

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm
from torch.optim import AdamW

# ==========================================

# 1. Setup & Config

# ==========================================

CHECKPOINT_DIR = '/content/drive/MyDrive/Altegrad/MolText_Retrieval_Checkpoints'

# Must match training config exactly

CONFIG = {

"mol_model_name": "ibm/MoLFormer-XL-both-10pct",
"text_model_name": "NeuML/pubmedbert-base-embeddings",
 "bi_encoder_path": "/content/drive/MyDrive/Altegrad/MolText_Retrieval_Checkpoints/best_model.pt",
"max_length": 128,
"projection_dim": 256,
"batch_size": 192,
"device": "cuda" if torch.cuda.is_available() else "cpu",
"temperature": 0.07,
"seed": 42
}

class CrossModalReranker(nn.Module):
    def __init__(self, bi_encoder_path, config):
        super().__init__()

        # 1. Load the Encoders (Architecture must match Bi-Encoder)
        self.mol_encoder = AutoModel.from_pretrained(config['mol_model_name'], trust_remote_code=True)
        self.text_encoder = AutoModel.from_pretrained(config['text_model_name'])

        mol_dim = self.mol_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size

        # 2. Pooling Layers (Must match Bi-Encoder)
        self.mol_pool = AttentivePooling(mol_dim)
        self.text_pool = AttentivePooling(text_dim)

        # 3. Load Pre-trained Weights from your Bi-Encoder
        print(f"Loading weights from {bi_encoder_path}...")
        state_dict = torch.load(bi_encoder_path, map_location=config['device'])
        if 'model_state_dict' in state_dict:
            state_dict = state_dict['model_state_dict']

        # We only want to load the encoder/pooling weights, not the projection heads
        # Filter and load weights carefully
        self.load_partial_state_dict(state_dict)

        # 4. The Classifier Head (The "Reranking" Logic)
        # Input is concatenated embeddings: mol_dim + text_dim
        self.classifier = nn.Sequential(
            nn.Linear(mol_dim + text_dim, 512),
            nn.BatchNorm1d(512),
            nn.Dropout(0.1),
            nn.ReLU(),
            nn.Linear(512, 1) # Outputs a single logit (score)
        )

    def load_partial_state_dict(self, state_dict):
        # Helper to load weights only for the encoders and pooling layers
        own_state = self.state_dict()
        for name, param in state_dict.items():
            if name in own_state:
                # Only load if shapes match (projection layers usually differ)
                if own_state[name].shape == param.shape:
                    own_state[name].copy_(param)

    def forward(self, mol_input, mol_mask, text_input, text_mask):
        # 1. Encode Molecule
        mol_out = self.mol_encoder(input_ids=mol_input, attention_mask=mol_mask)
        mol_emb = self.mol_pool(mol_out.last_hidden_state, mol_mask)

        # 2. Encode Text
        text_out = self.text_encoder(input_ids=text_input, attention_mask=text_mask)
        text_emb = self.text_pool(text_out.last_hidden_state, text_mask)

        # 3. Concatenate (Late Fusion)
        combined = torch.cat((mol_emb, text_emb), dim=1)

        # 4. Classify
        logits = self.classifier(combined)
        return logits

class RerankDataset(Dataset):
    def __init__(self, data, mol_tokenizer, text_tokenizer, max_len=128):
        """
        Args:
            data: Can be a Pandas DataFrame OR a Hugging Face Dataset.
                  Must contain columns/keys: 'smiles', 'text', 'label'.
        """
        self.data = data
        self.mol_tokenizer = mol_tokenizer
        self.text_tokenizer = text_tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 1. Access the row (Works for both HF Dataset and Pandas)
        row = self.data[idx]

        # 2. Tokenize SMILES
        mol_inputs = self.mol_tokenizer(
            str(row['smiles']),
            truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors="pt"
        )

        # 3. Tokenize Text
        text_inputs = self.text_tokenizer(
            str(row['text']),
            truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors="pt"
        )

        return {
            'mol_ids': mol_inputs['input_ids'].squeeze(0),
            'mol_mask': mol_inputs['attention_mask'].squeeze(0),
            'text_ids': text_inputs['input_ids'].squeeze(0),
            'text_mask': text_inputs['attention_mask'].squeeze(0),
            'labels': torch.tensor(row['label'], dtype=torch.float)
        }


class AttentivePooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, last_hidden_state, attention_mask):
        # 1. Calculate Attention Weights
        weights = self.attention(last_hidden_state)

        # 2. Mask Padding (so we don't attend to 0-padding)
        weights = weights.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)

        # 3. Softmax to get probabilities
        scores = F.softmax(weights, dim=1)

        # 4. Weighted Sum
        return torch.sum(last_hidden_state * scores, dim=1)

class ContrastiveModel(nn.Module):
    def __init__(self):
        super().__init__()

        # --- 1. Load Encoders ---
        # Trust remote code is needed for MoLFormer
        self.mol_encoder = AutoModel.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
        self.text_encoder = AutoModel.from_pretrained(CONFIG['text_model_name'])

        mol_dim = self.mol_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size

        # --- 2. Pooling Layers ---
        self.mol_pool = AttentivePooling(mol_dim)
        self.text_pool = AttentivePooling(text_dim)

        # --- 3. Projection Heads ---
        # Maps both modalities to the same shared embedding space (256 dim)
        self.mol_proj = nn.Sequential(
            nn.Linear(mol_dim, mol_dim),
            nn.BatchNorm1d(mol_dim),
            nn.GELU(),
            nn.Linear(mol_dim, CONFIG['projection_dim'])
        )

        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, text_dim),
            nn.BatchNorm1d(text_dim),
            nn.GELU(),
            nn.Linear(text_dim, CONFIG['projection_dim'])
        )

        # Learnable temperature parameter
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / CONFIG['temperature']))

    def forward(self, batch):
        # Note: This forward is mostly for training.
        # For mining/inference, we usually call encoder + pool + proj manually.

        # Molecule Branch
        mol_out = self.mol_encoder(input_ids=batch['mol_input_ids'], attention_mask=batch['mol_attention_mask'])
        mol_emb = self.mol_pool(mol_out.last_hidden_state, batch['mol_attention_mask'])
        mol_feat = self.mol_proj(mol_emb)

        # Text Branch
        text_out = self.text_encoder(input_ids=batch['text_input_ids'], attention_mask=batch['text_attention_mask'])
        text_emb = self.text_pool(text_out.last_hidden_state, batch['text_attention_mask'])
        text_feat = self.text_proj(text_emb)

        return mol_feat, text_feat

def mine_hard_negatives(model, df_raw):
    """
    1. Encodes all molecules and all descriptions in the dataframe using the Bi-Encoder.
    2. Finds the top (wrong) matches for each molecule.
    3. Returns a dataframe of pairs: (SMILES, Text, Label=1) and (SMILES, Text, Label=0).
    """
    print(f"--- Mining Hard Negatives from {len(df_raw)} samples ---")

    # Setup Tokenizers
    mol_tokenizer = AutoTokenizer.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
    text_tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model_name'])

    model.eval()

    # ==========================================
    # A. Encode All Texts (The "Bank")
    # ==========================================
    # Helper dataset just for encoding strings quickly
    class StringDataset(Dataset):
        def __init__(self, strings, tokenizer, max_len=128):
            self.strings = strings
            self.tokenizer = tokenizer
            self.max_len = max_len
        def __len__(self): return len(self.strings)
        def __getitem__(self, idx):
            txt = str(self.strings[idx])
            inputs = self.tokenizer(txt, truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt")
            return {'input_ids': inputs['input_ids'].squeeze(0), 'attention_mask': inputs['attention_mask'].squeeze(0)}

    print("Encoding Text Bank...")
    text_ds = StringDataset(df_raw['description'].tolist(), text_tokenizer)
    text_loader = DataLoader(text_ds, batch_size=CONFIG['batch_size']*2, shuffle=False, num_workers=2)

    bank_embs = []
    with torch.no_grad():
        for batch in tqdm(text_loader, desc="Encoding Texts"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            mask = batch['attention_mask'].to(CONFIG['device'])

            # Forward pass through Text Encoder
            out = model.text_encoder(input_ids, mask)
            vec = model.text_pool(out.last_hidden_state, mask)
            emb = model.text_proj(vec)
            bank_embs.append(F.normalize(emb, dim=-1).cpu())

    bank_embs = torch.cat(bank_embs, dim=0) # [N, 256]

    # ==========================================
    # B. Encode Molecules & Find Negatives
    # ==========================================
    print("Encoding Molecules & Finding Negatives...")
    mol_ds = StringDataset(df_raw['smiles'].tolist(), mol_tokenizer)
    mol_loader = DataLoader(mol_ds, batch_size=CONFIG['batch_size']*2, shuffle=False, num_workers=2)

    triplets = []

    # We iterate through molecules in batches
    start_idx = 0
    with torch.no_grad():
        for batch in tqdm(mol_loader, desc="Mining"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            mask = batch['attention_mask'].to(CONFIG['device'])

            # Encode Molecule Batch
            out = model.mol_encoder(input_ids, mask)
            vec = model.mol_pool(out.last_hidden_state, mask)
            emb = model.mol_proj(vec)
            mol_batch = F.normalize(emb, dim=-1).cpu() # [Batch, 256]

            # 1. Calculate Similarity: (Batch_Mols) @ (All_Texts.T)
            scores = torch.matmul(mol_batch, bank_embs.T)

            # 2. Get Top K indices (e.g., Top 10 to be safe)
            # We need enough candidates to find `num_hard_negatives` that are NOT the correct one.
            k_search = CONFIG['num_hard_negatives'] + 5
            top_k = torch.topk(scores, k=k_search, dim=1)
            indices = top_k.indices # [Batch, K]

            # 3. Process each molecule in the batch
            for i in range(len(mol_batch)):
                # The index of this molecule in the original dataframe
                true_idx = start_idx + i

                # Get the actual data
                smiles = df_raw.iloc[true_idx]['smiles']
                true_text = df_raw.iloc[true_idx]['description']

                # --- Add Positive Sample ---
                triplets.append({
                    'smiles': smiles,
                    'text': true_text,
                    'label': 1.0
                })

                # --- Add Negative Samples ---
                negatives_found = 0
                for candidate_idx in indices[i]:
                    candidate_idx = candidate_idx.item()

                    # Skip if the model accidentally retrieved the correct answer (we don't want to punish it for that)
                    if candidate_idx == true_idx:
                        continue

                    neg_text = df_raw.iloc[candidate_idx]['description']

                    triplets.append({
                        'smiles': smiles,
                        'text': neg_text,
                        'label': 0.0
                    })

                    negatives_found += 1
                    if negatives_found >= CONFIG['num_hard_negatives']:
                        break

            start_idx += len(batch['input_ids'])

    return pd.DataFrame(triplets)


def train_reranker(ds_train, ds_val):
    # Setup Tokenizers
    mol_tokenizer = AutoTokenizer.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
    text_tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model_name'])

    # Load Bi-Encoder for mining
    print(f"Loading Bi-Encoder from {CONFIG['bi_encoder_path']}...")
    bi_model = ContrastiveModel().to(CONFIG['device'])
    state_dict = torch.load(CONFIG['bi_encoder_path'], map_location=CONFIG['device'])
    if 'model_state_dict' in state_dict: state_dict = state_dict['model_state_dict']
    bi_model.load_state_dict(state_dict)

    # --- 1. Mining Hard Negatives ---
    # We must convert HF datasets to Pandas for the mining logic to work easily
    # (The mining function we wrote earlier expects a dataframe/list)
    print("Converting HF Datasets to Pandas for Mining...")
    df_train_raw = ds_train.to_pandas()
    df_val_raw = ds_val.to_pandas()

    print("Mining Negatives for TRAIN set...")
    df_train_mined = mine_hard_negatives(bi_model, df_train_raw)

    print("Mining Negatives for VAL set...")
    # We also mine negatives for validation to measure accuracy properly
    df_val_mined = mine_hard_negatives(bi_model, df_val_raw)

    # Free memory
    del bi_model
    torch.cuda.empty_cache()

    # --- 2. Initialize Custom Reranker ---
    reranker = CrossModalReranker(CONFIG['bi_encoder_path'], CONFIG).to(CONFIG['device'])

    # --- 3. Datasets & Loaders ---
    # Now we pass the MINED dataframes (which contain labels 0 and 1)
    train_ds = RerankDataset(df_train_mined, mol_tokenizer, text_tokenizer)
    val_ds = RerankDataset(df_val_mined, mol_tokenizer, text_tokenizer)

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    optimizer = AdamW(reranker.parameters(), lr=2e-5)
    criterion = nn.BCEWithLogitsLoss()

    # --- 4. Training Loop ---
    best_val_loss = float('inf')

    for epoch in range(CONFIG['epochs']):
        # --- TRAIN ---
        reranker.train()
        train_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")

        for batch in loop:
            mol_ids = batch['mol_ids'].to(CONFIG['device'])
            mol_mask = batch['mol_mask'].to(CONFIG['device'])
            text_ids = batch['text_ids'].to(CONFIG['device'])
            text_mask = batch['text_mask'].to(CONFIG['device'])
            labels = batch['labels'].to(CONFIG['device'])

            optimizer.zero_grad()
            logits = reranker(mol_ids, mol_mask, text_ids, text_mask).squeeze(-1)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        # --- VALIDATION ---
        reranker.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
                mol_ids = batch['mol_ids'].to(CONFIG['device'])
                mol_mask = batch['mol_mask'].to(CONFIG['device'])
                text_ids = batch['text_ids'].to(CONFIG['device'])
                text_mask = batch['text_mask'].to(CONFIG['device'])
                labels = batch['labels'].to(CONFIG['device'])

                logits = reranker(mol_ids, mol_mask, text_ids, text_mask).squeeze(-1)
                loss = criterion(logits, labels)
                val_loss += loss.item()

                # Calculate Accuracy (threshold 0.5)
                preds = torch.sigmoid(logits) > 0.5
                correct += (preds == (labels > 0.5)).sum().item()
                total += labels.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / total
        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # Save Best Model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            save_path = os.path.join(CONFIG['save_dir'], "reranker_best.pt")
            torch.save(reranker.state_dict(), save_path)
            print(f"--> Best model saved to {save_path}")

# Run it
if __name__ == "__main__":
    train_reranker(ds_train, ds_val)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
import gc

# ==========================================
# 1. Config (Must match your Bi-Encoder)
# ==========================================

CONFIG = {
    "mol_model_name": "ibm/MoLFormer-XL-both-10pct",
    "text_model_name": "NeuML/pubmedbert-base-embeddings",
    "bi_encoder_path": "/content/drive/MyDrive/Altegrad/MolText_Retrieval_Checkpoints/best_model.pt",
    "save_dir": "/content/drive/MyDrive/Altegrad/Reranker_Checkpoints",
    "max_length": 128,
    "batch_size": 128, # Can usually be higher for reranker
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "epochs": 5, # Rerankers converge quickly
    "lr": 5e-5,
    "num_hard_negatives": 3 # Number of negatives per positive to mine
}
os.makedirs(CONFIG['save_dir'], exist_ok=True)

# ==========================================
# 2. Shared Components
# ==========================================

class AttentivePooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, last_hidden_state, attention_mask):
        weights = self.attention(last_hidden_state)
        weights = weights.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)
        scores = F.softmax(weights, dim=1)
        return torch.sum(last_hidden_state * scores, dim=1)

# ==========================================
# 3. The Reranker Model
# ==========================================

class CrossModalReranker(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.mol_encoder = AutoModel.from_pretrained(config['mol_model_name'], trust_remote_code=True)
        self.text_encoder = AutoModel.from_pretrained(config['text_model_name'])

        mol_dim = self.mol_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size

        self.mol_pool = AttentivePooling(mol_dim)
        self.text_pool = AttentivePooling(text_dim)

        # Improved Classifier: "Residual MLP" to help gradient flow
        combined_dim = mol_dim + text_dim
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Linear(512, 1)
        )

        # Load weights intelligently
        self.load_bi_encoder_weights(config['bi_encoder_path'], config['device'])

    def load_bi_encoder_weights(self, path, device):
        print(f"Loading Bi-Encoder weights from {path}...")
        checkpoint = torch.load(path, map_location=device)
        state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint

        # Filter and load only the Encoders and Pooling layers
        # We DO NOT load the projection heads or existing classifiers
        own_state = self.state_dict()
        for name, param in state_dict.items():
            if name in own_state and param.shape == own_state[name].shape:
                own_state[name].copy_(param)
        print("✅ Encoders initialized from Bi-Encoder.")

    def forward(self, mol_input, mol_mask, text_input, text_mask):
        mol_out = self.mol_encoder(input_ids=mol_input, attention_mask=mol_mask)
        mol_emb = self.mol_pool(mol_out.last_hidden_state, mol_mask)

        text_out = self.text_encoder(input_ids=text_input, attention_mask=text_mask)
        text_emb = self.text_pool(text_out.last_hidden_state, text_mask)

        # Late Fusion
        combined = torch.cat((mol_emb, text_emb), dim=1)
        logits = self.classifier(combined)
        return logits

# ==========================================
# 4. Hard Negative Mining (Memory Safe)
# ==========================================

class ContrastiveModelForMining(nn.Module):
    """Temporary class just to load the Bi-Encoder for mining"""
    def __init__(self):
        super().__init__()
        self.mol_encoder = AutoModel.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
        self.text_encoder = AutoModel.from_pretrained(CONFIG['text_model_name'])
        mol_dim = self.mol_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size
        self.mol_pool = AttentivePooling(mol_dim)
        self.text_pool = AttentivePooling(text_dim)
        # We define projections just to match state_dict keys, we don't use them if not needed
        self.mol_proj = nn.Linear(mol_dim, CONFIG.get('projection_dim', 256))
        self.text_proj = nn.Linear(text_dim, CONFIG.get('projection_dim', 256))

    def forward(self, batch):
        pass

def mine_hard_negatives(df_raw, split_name="train"):
    print(f"\n--- Mining Hard Negatives for {split_name} ({len(df_raw)} samples) ---")

    # 1. Load Bi-Encoder
    model = ContrastiveModelForMining().to(CONFIG['device'])
    try:
        # Strict=False allows us to ignore mismatches in projection heads if dimensions changed
        state_dict = torch.load(CONFIG['bi_encoder_path'], map_location=CONFIG['device'])
        if 'model_state_dict' in state_dict: state_dict = state_dict['model_state_dict']
        model.load_state_dict(state_dict, strict=False)
    except Exception as e:
        print(f"Warning: Could not load full state dict ({e}). Ensure architecture matches.")

    model.eval()
    mol_tokenizer = AutoTokenizer.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
    text_tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model_name'])

    # 2. Embed All Texts (The "Bank")
    # We use a simple list dataset for speed
    texts = df_raw['description'].tolist()
    text_embs = []

    # Process texts in batches
    batch_size = CONFIG['batch_size']
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding Texts"):
        batch_texts = texts[i:i+batch_size]
        inputs = text_tokenizer(batch_texts, truncation=True, padding=True, max_length=CONFIG['max_length'], return_tensors="pt").to(CONFIG['device'])
        with torch.no_grad():
            out = model.text_encoder(**inputs)
            pool = model.text_pool(out.last_hidden_state, inputs['attention_mask'])
            # Only if your bi-encoder uses projection for similarity, normalize here.
            # Assuming bi-encoder used Normalized Projections:
            if hasattr(model, 'text_proj'):
                 # Note: The loaded state_dict might have different proj layers if you changed code.
                 # Safest is to use the raw pooled embeddings for mining if unsure,
                 # BUT for accuracy, we should use the projection if the bi-encoder was trained with it.
                 # Here we assume the mining model structure matches the training one.
                 pass
            text_embs.append(F.normalize(pool, dim=1).cpu()) # Move to CPU to save GPU

    bank = torch.cat(text_embs, dim=0) # [N, Dim] on CPU

    # 3. Mine Logic
    triplets = []
    smiles_list = df_raw['smiles'].tolist()

    for i in tqdm(range(0, len(smiles_list), batch_size), desc="Mining"):
        batch_smiles = smiles_list[i:i+batch_size]
        inputs = mol_tokenizer(batch_smiles, truncation=True, padding=True, max_length=CONFIG['max_length'], return_tensors="pt").to(CONFIG['device'])

        with torch.no_grad():
            out = model.mol_encoder(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'])
            pool = model.mol_pool(out.last_hidden_state, inputs['attention_mask'])
            mol_batch = F.normalize(pool, dim=1).cpu() # Move to CPU

        # Similarity on CPU (Safe from OOM)
        scores = torch.matmul(mol_batch, bank.T)

        # We want the top K hardest negatives
        # We fetch K+10 because the "correct" answer is likely in the top results
        top_k = torch.topk(scores, k=CONFIG['num_hard_negatives'] + 10, dim=1)

        indices = top_k.indices.numpy()

        for idx_in_batch, query_indices in enumerate(indices):
            true_idx = i + idx_in_batch

            # 1. Add Positive
            triplets.append({
                'smiles': smiles_list[true_idx],
                'text': texts[true_idx],
                'label': 1.0
            })

            # 2. Add Negatives
            added = 0
            for cand_idx in query_indices:
                if cand_idx == true_idx: continue # Skip self

                triplets.append({
                    'smiles': smiles_list[true_idx],
                    'text': texts[cand_idx],
                    'label': 0.0
                })
                added += 1
                if added >= CONFIG['num_hard_negatives']: break

    del model, bank, text_embs
    torch.cuda.empty_cache()
    gc.collect()

    return pd.DataFrame(triplets)

# ==========================================
# 5. Dataset & Train Loop
# ==========================================

class RerankDataset(Dataset):
    def __init__(self, df, mol_tok, text_tok):
        self.data = df
        self.mol_tok = mol_tok
        self.text_tok = text_tok

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        m = self.mol_tok(str(row['smiles']), truncation=True, padding='max_length', max_length=CONFIG['max_length'], return_tensors="pt")
        t = self.text_tok(str(row['text']), truncation=True, padding='max_length', max_length=CONFIG['max_length'], return_tensors="pt")
        return {
            'mol_ids': m['input_ids'].squeeze(0), 'mol_mask': m['attention_mask'].squeeze(0),
            'text_ids': t['input_ids'].squeeze(0), 'text_mask': t['attention_mask'].squeeze(0),
            'labels': torch.tensor(row['label'], dtype=torch.float)
        }

def train_reranker(ds_train, ds_val):
    # Convert HF datasets to Pandas for easier mining handling
    print("Preparing Data...")
    df_train = ds_train.to_pandas() if hasattr(ds_train, 'to_pandas') else pd.DataFrame(ds_train)
    df_val = ds_val.to_pandas() if hasattr(ds_val, 'to_pandas') else pd.DataFrame(ds_val)

    # Mine Negatives
    train_mined = mine_hard_negatives(df_train, "Train")
    val_mined = mine_hard_negatives(df_val, "Val") # Optional: usually we validate on random or just positive pairs, but hard neg val is good too

    mol_tokenizer = AutoTokenizer.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
    text_tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model_name'])

    train_ds = RerankDataset(train_mined, mol_tokenizer, text_tokenizer)
    val_ds = RerankDataset(val_mined, mol_tokenizer, text_tokenizer)

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    model = CrossModalReranker(CONFIG).to(CONFIG['device'])
    optimizer = AdamW(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.BCEWithLogitsLoss()

    best_acc = 0.0

    print("--- Starting Reranker Training ---")
    for epoch in range(CONFIG['epochs']):
        model.train()
        train_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

        for batch in loop:
            batch = {k: v.to(CONFIG['device']) for k, v in batch.items()}

            optimizer.zero_grad()
            logits = model(batch['mol_ids'], batch['mol_mask'], batch['text_ids'], batch['text_mask']).squeeze(-1)
            loss = criterion(logits, batch['labels'])
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(CONFIG['device']) for k, v in batch.items()}
                logits = model(batch['mol_ids'], batch['mol_mask'], batch['text_ids'], batch['text_mask']).squeeze(-1)
                preds = (torch.sigmoid(logits) > 0.5).float()
                correct += (preds == batch['labels']).sum().item()
                total += len(batch['labels'])

        acc = correct / total
        print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), os.path.join(CONFIG['save_dir'], "reranker_best.pt"))
            print("--> Model Saved")

if __name__ == "__main__":
    # Ensure you pass your datasets here
    # ds_train and ds_val must be available from your other script
    train_reranker(ds_train, ds_val)

## step 3 : inference retrieval + reranking

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# ==========================================
# 1. Setup & Configuration
# ==========================================

# Adjust these paths to match your actual files
CHECKPOINT_DIR_BI = '/content/drive/MyDrive/Altegrad/MolText_Retrieval_Checkpoints'
CHECKPOINT_DIR_RERANK = '/content/drive/MyDrive/Altegrad/Reranker_Checkpoints'

CONFIG = {
    # --- Bi-Encoder Config ---
    "mol_model_name": "ibm/MoLFormer-XL-both-10pct",
    "text_model_name": "NeuML/pubmedbert-base-embeddings",
    "max_length": 128,
    "projection_dim": 256,
    "batch_size": 128, # Inference batch size
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # --- Reranking Strategy ---
    "top_k": 50,  # Retrieve top 50 candidates to rerank

    # --- Model Weights ---
    "bi_encoder_weights": os.path.join(CHECKPOINT_DIR_BI, "best_model.pt"),
    "reranker_weights": os.path.join(CHECKPOINT_DIR_RERANK, "reranker_best.pt")
}

# ==========================================
# 2. Model Architectures (Must match Training!)
# ==========================================

class AttentivePooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, last_hidden_state, attention_mask):
        weights = self.attention(last_hidden_state)
        weights = weights.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)
        scores = F.softmax(weights, dim=1)
        return torch.sum(last_hidden_state * scores, dim=1)

class ContrastiveModel(nn.Module):
    """The Bi-Encoder for Stage 1 Retrieval"""
    def __init__(self):
        super().__init__()
        self.mol_encoder = AutoModel.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
        self.text_encoder = AutoModel.from_pretrained(CONFIG['text_model_name'])

        mol_dim = self.mol_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size

        self.mol_pool = AttentivePooling(mol_dim)
        self.text_pool = AttentivePooling(text_dim)

        self.mol_proj = nn.Sequential(
            nn.Linear(mol_dim, mol_dim), nn.BatchNorm1d(mol_dim), nn.GELU(),
            nn.Linear(mol_dim, CONFIG['projection_dim'])
        )
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, text_dim), nn.BatchNorm1d(text_dim), nn.GELU(),
            nn.Linear(text_dim, CONFIG['projection_dim'])
        )

        # --- FIX: This line was missing ---
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

class CrossModalReranker(nn.Module):
    """The Reranker for Stage 2 Refinement"""
    def __init__(self):
        super().__init__()
        self.mol_encoder = AutoModel.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
        self.text_encoder = AutoModel.from_pretrained(CONFIG['text_model_name'])

        mol_dim = self.mol_encoder.config.hidden_size
        text_dim = self.text_encoder.config.hidden_size

        self.mol_pool = AttentivePooling(mol_dim)
        self.text_pool = AttentivePooling(text_dim)

        # Classifier Head (Must match training code exactly)
        combined_dim = mol_dim + text_dim
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 1024), nn.BatchNorm1d(1024), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.GELU(),
            nn.Linear(512, 1)
        )

    def forward(self, mol_input, mol_mask, text_input, text_mask):
        mol_out = self.mol_encoder(input_ids=mol_input, attention_mask=mol_mask)
        mol_emb = self.mol_pool(mol_out.last_hidden_state, mol_mask)

        text_out = self.text_encoder(input_ids=text_input, attention_mask=text_mask)
        text_emb = self.text_pool(text_out.last_hidden_state, text_mask)

        combined = torch.cat((mol_emb, text_emb), dim=1)
        return self.classifier(combined)

# ==========================================
# 3. Helper Datasets
# ==========================================

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = texts
        self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        t = str(self.texts[idx])
        inputs = self.tokenizer(t, truncation=True, padding='max_length', max_length=CONFIG['max_length'], return_tensors="pt")
        return {'input_ids': inputs['input_ids'].squeeze(0), 'attention_mask': inputs['attention_mask'].squeeze(0)}

class MolDataset(Dataset):
    def __init__(self, smiles, tokenizer):
        self.smiles = smiles
        self.tokenizer = tokenizer
    def __len__(self): return len(self.smiles)
    def __getitem__(self, idx):
        s = str(self.smiles[idx])
        inputs = self.tokenizer(s, truncation=True, padding='max_length', max_length=CONFIG['max_length'], return_tensors="pt")
        return {'input_ids': inputs['input_ids'].squeeze(0), 'attention_mask': inputs['attention_mask'].squeeze(0)}

# ==========================================
# 4. Main Inference Function
# ==========================================

def generate_submission(ds_train, ds_test, output_file):
    print("--- Setup ---")
    mol_tokenizer = AutoTokenizer.from_pretrained(CONFIG['mol_model_name'], trust_remote_code=True)
    text_tokenizer = AutoTokenizer.from_pretrained(CONFIG['text_model_name'])

    # Safe convert to list
    df_train = ds_train.to_pandas() if hasattr(ds_train, 'to_pandas') else pd.DataFrame(ds_train)
    df_test = ds_test.to_pandas() if hasattr(ds_test, 'to_pandas') else pd.DataFrame(ds_test)

    bank_texts = df_train['description'].tolist()
    test_smiles = df_test['smiles'].tolist()
    test_ids = df_test['id'].tolist() if 'id' in df_test.columns else df_test['ID'].tolist()

    # ---------------------------------------------------------
    # PART A: BI-ENCODER (Global Retrieval)
    # ---------------------------------------------------------
    print("\n[Stage 1] Loading Bi-Encoder...")
    bi_model = ContrastiveModel().to(CONFIG['device'])
    ckpt = torch.load(CONFIG['bi_encoder_weights'], map_location=CONFIG['device'])
    bi_model.load_state_dict(ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt)
    bi_model.eval()

    # 1. Embed Text Bank
    print(f"Indexing {len(bank_texts)} texts...")
    text_ds = TextDataset(bank_texts, text_tokenizer)
    text_loader = DataLoader(text_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    bank_embs = []
    with torch.no_grad():
        for batch in tqdm(text_loader, desc="Bi-Enc Text"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            mask = batch['attention_mask'].to(CONFIG['device'])

            # Encode -> Pool -> Project -> Normalize
            out = bi_model.text_encoder(input_ids, mask)
            vec = bi_model.text_pool(out.last_hidden_state, mask)
            emb = bi_model.text_proj(vec)
            bank_embs.append(F.normalize(emb, dim=-1).cpu())

    bank_embs = torch.cat(bank_embs, dim=0) # [N_Texts, Dim]

    # 2. Embed Test Molecules
    print(f"Encoding {len(test_smiles)} query molecules...")
    mol_ds = MolDataset(test_smiles, mol_tokenizer)
    mol_loader = DataLoader(mol_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    query_embs = []
    with torch.no_grad():
        for batch in tqdm(mol_loader, desc="Bi-Enc Mol"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            mask = batch['attention_mask'].to(CONFIG['device'])

            out = bi_model.mol_encoder(input_ids, mask)
            vec = bi_model.mol_pool(out.last_hidden_state, mask)
            emb = bi_model.mol_proj(vec)
            query_embs.append(F.normalize(emb, dim=-1).cpu())

    query_embs = torch.cat(query_embs, dim=0)

    # 3. Initial Retrieval
    print("Computing dot products...")
    # Matrix mult on CPU to avoid OOM
    scores = torch.matmul(query_embs, bank_embs.T)
    top_k_indices = torch.topk(scores, k=CONFIG['top_k'], dim=1).indices

    # Clean up Bi-Encoder
    del bi_model, query_embs, bank_embs
    torch.cuda.empty_cache()

    # ---------------------------------------------------------
    # PART B: RERANKER (Local Refinement)
    # ---------------------------------------------------------
    print("\n[Stage 2] Loading Reranker...")
    reranker = CrossModalReranker().to(CONFIG['device'])
    ckpt = torch.load(CONFIG['reranker_weights'], map_location=CONFIG['device'])
    # Handle state dict mismatch if loaded incorrectly
    state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    reranker.load_state_dict(state_dict)
    reranker.eval()

    final_results = []

    print(f"Reranking top {CONFIG['top_k']} candidates for each query...")

    # We iterate 1-by-1 or small batches. For simplicity: 1-by-1 query
    for i in tqdm(range(len(test_smiles)), desc="Reranking"):
        query_smi = test_smiles[i]
        candidate_idxs = top_k_indices[i].numpy()

        # Prepare Batch for Reranker: [Query] repeated k times, [Candidates]
        batch_smiles = [query_smi] * len(candidate_idxs)
        batch_texts = [bank_texts[idx] for idx in candidate_idxs]

        # Tokenize on the fly
        mol_inputs = mol_tokenizer(batch_smiles, truncation=True, padding='max_length', max_length=CONFIG['max_length'], return_tensors="pt")
        text_inputs = text_tokenizer(batch_texts, truncation=True, padding='max_length', max_length=CONFIG['max_length'], return_tensors="pt")

        mol_ids = mol_inputs['input_ids'].to(CONFIG['device'])
        mol_mask = mol_inputs['attention_mask'].to(CONFIG['device'])
        text_ids = text_inputs['input_ids'].to(CONFIG['device'])
        text_mask = text_inputs['attention_mask'].to(CONFIG['device'])

        with torch.no_grad():
            logits = reranker(mol_ids, mol_mask, text_ids, text_mask).squeeze(-1)

        # The best candidate index (0 to k-1)
        best_local_idx = torch.argmax(logits).item()

        # Map back to global text index
        best_global_idx = candidate_idxs[best_local_idx]
        best_text = bank_texts[best_global_idx]

        final_results.append({
            "ID": test_ids[i],
            "description": best_text
        })

    # ---------------------------------------------------------
    # PART C: SAVING
    # ---------------------------------------------------------
    df_submission = pd.DataFrame(final_results)
    df_submission.to_csv(output_file, index=False)
    print(f"✅ Submission saved to {output_file}")
    print(df_submission.head())

if __name__ == "__main__":
    # Assuming ds_train and ds_test are available in your notebook
    # If not, load them here: ds_train = pd.read_csv("...")

    OUTPUT_FILE = '/content/drive/MyDrive/Altegrad/submission_reranked.csv'
    generate_submission(ds_train, ds_test, OUTPUT_FILE)